In [0]:
import requests 
import pandas as pd
import io
from pyspark.sql import *
from pyspark.sql.functions import lit, current_timestamp




In [0]:
TABLE = "project_mobility.bronze.taxi_trips_raw"

def already_loaded(year, month):
    try:
        count = spark.sql(f"""
            SELECT COUNT(*) as cnt FROM {TABLE}
            WHERE year = {year} AND month = {month}
        """).collect()[0]['cnt']
        return count > 0
    except:
        return False  # table doesn't exist yet

months = [
    '2025-01', '2025-02', '2025-03', '2025-04', '2025-05', '2025-06',
    '2025-07', '2025-08', '2025-09', '2025-10', '2025-11', '2025-12'
]

for month in months:
    year_val = int(month[:4])
    month_val = int(month[5:])

    if already_loaded(year_val, month_val):
        print(f"  {month} already loaded — skipping")
        continue

    try:
        url = f"https://d37ci6vzurychx.cloudfront.net/trip-data/green_tripdata_{month}.parquet"
        print(f"Loading {month}...")

        response = requests.get(url)
        response.raise_for_status()

        df = spark.createDataFrame(pd.read_parquet(io.BytesIO(response.content)))

        df = df.withColumns({"source_file": lit(url),"ingestion_timestamp": current_timestamp(),"year": lit(year_val),"month": lit(month_val)})

        row_count = df.count()

        df.write.mode("append") \
            .partitionBy("year", "month") \
            .option("mergeSchema", "true") \
            .saveAsTable(TABLE)

        print(f"  {month} done — {row_count} rows")

    except Exception as e:
        print(f"  {month} FAILED: {e}")